# Success intervals — Wilson score

Per-backend success rate with small-n Wilson confidence intervals.

In [ ]:
from analysis import load_runs

# Pin matplotlib (Agg) + RNG seeds for reproducible, headless figures.
plt = load_runs.pin_style(seed=0)

# Load real runs if any exist, else the deterministic synthetic dataset.
records = load_runs.load_all()
backends = load_runs.backend_names(records)
STRATEGY = "zero_shot"  # the held-fixed strategy for the cross-backend tests
print(f"{len(records)} runs, backends={backends}")

In [ ]:
from analysis import stats as st
from analysis.aggregate import aggregate_runs, build_outcome_matrix

summaries = aggregate_runs(records)
ids, matrix = build_outcome_matrix(summaries, backends=backends, strategy=STRATEGY)
n = len(ids)
cis = []
for j, b in enumerate(backends):
    succ = sum(row[j] for row in matrix)
    cis.append(st.wilson_ci(succ, n, backend=b))
for ci in cis:
    print(
        f"{ci.backend}: {ci.successes}/{ci.n} = {ci.proportion:.2f} "
        f"[{ci.ci_low:.2f}, {ci.ci_high:.2f}] ({ci.confidence:.0%} CI)"
    )

In [ ]:
props = [c.proportion for c in cis]
lo = [c.proportion - c.ci_low for c in cis]
hi = [c.ci_high - c.proportion for c in cis]
fig, ax = plt.subplots()
ax.errorbar(backends, props, yerr=[lo, hi], fmt="o", capsize=6, color="darkorange")
ax.set_ylim(0, 1)
ax.set_ylabel("success rate")
ax.set_title(f"Wilson {cis[0].confidence:.0%} CIs for success rate ({STRATEGY})")
plt.setp(ax.get_xticklabels(), rotation=20, ha="right")
load_runs.save_figure(fig, "05_success_intervals", "wilson_intervals")
fig

In [ ]:
from IPython.display import Markdown

lines = [
    f"| {c.backend} | {c.successes}/{c.n} | {c.proportion:.2f} | "
    f"[{c.ci_low:.2f}, {c.ci_high:.2f}] |"
    for c in cis
]
Markdown(
    f"### Summary — success intervals (Wilson, {cis[0].confidence:.0%})\n\n"
    "| backend | solved | rate | CI |\n|---|---|---|---|\n" + "\n".join(lines)
)